In [30]:
%pip install requests
%pip install bs4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import sys

import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import time as t

In [32]:
def extract_state(html):
    soup = BeautifulSoup(html, "html.parser")

    def val(name):
        tag = soup.find("input", {"name": name})
        return tag["value"] if tag else None

    return {
        "__VIEWSTATE": val("__VIEWSTATE"),
        "__VIEWSTATEGENERATOR": val("__VIEWSTATEGENERATOR"),
        "__EVENTVALIDATION": val("__EVENTVALIDATION"),
    }

In [33]:
def send_request(url, payload, method):
    if method.upper() == "GET":
        r = session.get(url, params=payload)
    else:
        r = session.post(url, data=payload)
    # print("Logged in:", r.status_code)
    # print("Session cookies:", session.cookies.get_dict())

    return r


In [34]:
def get_user_tee_time_days(session, authenticated_url):
    global user_tee_times
    response = session.get(authenticated_url + "MyTeeTimes.aspx")
    soup = BeautifulSoup(response.text, "html.parser")
    current_year = datetime.now().year
    for card in soup.find_all("div", class_="card"):
        card_text = card.find("p", class_="card-text")
        if card_text and "," in card_text.text:
            date_str = card_text.text.split(",")[0]  # e.g., "Sat Apr 18"
            try:
                # Parse with current year
                parsed_date = datetime.strptime(f"{date_str} {current_year}", "%a %b %d %Y")
                formatted_date = parsed_date.strftime("%m/%d/%Y").lstrip('0')
                user_tee_times.append(formatted_date)
            except ValueError:
                continue


In [35]:

def load_booking_requests(csv_path):
    global user_tee_times
    df = pd.read_csv(csv_path)

    valid_rows = []
    for _, row in df.iterrows():
        try:
            date_obj = datetime.strptime(row['Date'], '%m/%d/%Y')
            formatted_date = date_obj.strftime("%m/%d/%Y").lstrip('0')

            if formatted_date in user_tee_times:
                print(f"Skipping already booked date: {formatted_date}")
                continue

            time_obj = datetime.strptime(row['Time'].strip().lower(), '%I:%M%p')
            combined_dt = datetime.combine(date_obj.date(), time_obj.time())
        
            if str(row.get('Accept Earlier Time?', 'N')).strip().upper() == 'N':
                accept_earlier_time_obj = False
            else:
                accept_earlier_time_obj = True

            if str(row.get('Accept Later Time?', 'N')).strip().upper() == 'N':
                accept_later_time_obj = False
            else:
                accept_later_time_obj = True
            
            if str(row.get('Pair up with others?', 'N')).strip().upper() == 'N':
                pair_with_others_obj = False
            else:
                pair_with_others_obj = True

            if combined_dt < datetime.now():
                print(f"Skipping past booking: {row['Date']} {row['Time']}")
                continue
            if date_obj.date() > datetime.now().date() + timedelta(days=7):
                print(f"Skipping dates further than 1 week away: {row['Date']}")
                continue

            valid_rows.append({
                'date': date_obj,
                'time': time_obj.time(),
                'players': int(row['Number of Players']),
                'pair_with_others': pair_with_others_obj,
                'accept_earlier_time': accept_earlier_time_obj,
                'accept_later_time': accept_later_time_obj
            })
        except Exception as e:
            print(f"Skipping invalid row {row} — Error: {e}")

    return valid_rows

In [36]:


session = requests.Session()

initial_resp = session.get("https://booking.proshopteetimes.com/selectplayers.aspx?CourseID=132")
url = initial_resp.url

login_resp = session.get(url)
state = extract_state(login_resp.text)

username = "pmiles2559@gmail.com"
password = "3749"

payload = {
    "__EVENTTARGET": "",
    "__EVENTARGUMENT": "",
    "__VIEWSTATE": state["__VIEWSTATE"],
    "__VIEWSTATEGENERATOR": state["__VIEWSTATEGENERATOR"],
    "__EVENTVALIDATION": state["__EVENTVALIDATION"],
    "txtUserName": username,
    "Password": password,
    "btnOK": "Sign In"
}


logged_in_sesssion = send_request(url, payload, "POST")
authenticated_url = logged_in_sesssion.url.rsplit('/', 1)[0] + '/'
# print(f"Authenticated URL: {authenticated_url}")

# Find user tee times to avoid booking same day 
user_tee_times = []
print (f" before: {user_tee_times}")
get_user_tee_time_days(session, authenticated_url)
print(f" after: {user_tee_times}")

booking_requests = load_booking_requests('C:/Users/Patrick/Documents/repos/tee_time_scheduler/tee_time_booking - Copy.csv')
# print(booking_requests)

if not booking_requests:
    print ("Invalid or empty booking data.")
    sys.exit(0)





 before: []
 after: ['4/18/2026']


In [ ]:
for req in booking_requests:

    
    state = extract_state(session.get(authenticated_url + "SelectPlayers.aspx").text)

    #SELECT PLAYERS POST REQUEST
    select_players_payload = {
        "__EVENTTARGET": f"Button{req['players']}",
        "__EVENTARGUMENT": "",
        "__VIEWSTATE": state["__VIEWSTATE"],
        "__VIEWSTATEGENERATOR": state["__VIEWSTATEGENERATOR"],
        "__EVENTVALIDATION": state["__EVENTVALIDATION"]
    }

    send_request(authenticated_url + "SelectPlayers.aspx", select_players_payload, "POST")

    formatted_dt = req['date'].strftime('%m/%d/%Y').lstrip('0')
    # print(authenticated_url + "SelectTime.aspx" + f"?dt={formatted_dt}")
    

    #SELECT DATE REQUEST

    state = extract_state(session.get(authenticated_url + "SelectDate.aspx").text)

    #SELECT TIME REQUEST

    # print (f"Sending GET request to {authenticated_url}SelectTime.aspx ?dt={formatted_dt} with params: {select_time_params}")
    select_time_response = send_request(authenticated_url + "SelectTime.aspx" + f"?dt={formatted_dt}", None, "GET")
    # print(select_time_response.text)

    state = extract_state(session.get(authenticated_url + "SelectTime.aspx" + f"?dt={formatted_dt}").text)

    #TEE TIME SELECTED
    soup = BeautifulSoup(select_time_response.text, "html.parser")
    ttids = []
    for link in soup.find_all("a", href=True):
        href = link['href']
        if "TeeTimeSelected.aspx?TTID=" in href:
            time_tag = link.find("h4", class_="card-title")
            time = time_tag.text.split()[0] if time_tag else None
            not_available = link.find("p", string=lambda text: "Not Available" in text if text else False)
            available = not_available is None

            if available:
                parsed_time = datetime.strptime(time.strip().lower(), '%I:%M%p').time()
                if parsed_time <= req['time'] and req['accept_earlier_time'] or parsed_time >= req['time'] and req['accept_later_time']:
                    ttids.append({
                        'href': href,
                    })
    # print(ttids)
    # print(ttids[0]['href'])
    
    tee_time_selected_response = send_request(authenticated_url + ttids[0]['href'], None, "GET")
    # print(tee_time_selected_response.text)

    state = extract_state(session.get(authenticated_url + ttids[0]['href']).text)

    confirm_booking_payload = {
        "__EVENTTARGET": "btnOk",
        "__EVENTARGUMENT": "",
        "__VIEWSTATE": state["__VIEWSTATE"],
        "__VIEWSTATEGENERATOR": state["__VIEWSTATEGENERATOR"],
        "__EVENTVALIDATION": state["__EVENTVALIDATION"]
    }

    confirm_booking_response = send_request(authenticated_url + "ConfirmBooking.aspx", confirm_booking_payload, "POST")
    # print(confirm_booking_response.text)


    state = extract_state(session.get(authenticated_url + "BookingConfirmed.aspx").text)
    booking_confirmed_response = send_request(authenticated_url + "BookingConfirmed.aspx", None, "GET")
    # print(booking_confirmed_response.text)